# 10 · Capstone Project — Retail Sales Analysis

**Goal:** combine everything from notebooks 01–09 (creation, indexing, cleaning, filtering,
groupby, merging, datetime handling, and I/O/reshaping) into one realistic analysis.

We'll analyze a small synthetic retail dataset: **orders**, **products**, and **customers** —
a classic multi-table scenario.

## Step 1 — Build the (synthetic) source data

In a real project this would be `pd.read_csv(...)`. Here we construct it directly so the
notebook is fully self-contained and reproducible.

In [1]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(seed=42)

customers = pd.DataFrame({
    "customer_id": range(1, 9),
    "customer_name": ["Alice", "Bob", "Charlie", "Diana", "Evan", "Fiona", "George", "Hana"],
    "city": ["NYC", "LA", "NYC", "Chicago", "LA", "NYC", "Houston", "Chicago"]
})

products = pd.DataFrame({
    "product_id": range(101, 106),
    "product_name": ["Widget", "Gadget", "Gizmo", "Doohickey", "Thingamajig"],
    "category": ["Tools", "Tech", "Tech", "Tools", "Misc"],
    "unit_price": [9.99, 24.99, 19.99, 14.99, 5.99]
})

n_orders = 40
orders = pd.DataFrame({
    "order_id": range(1, n_orders + 1),
    "customer_id": rng.integers(1, 9, n_orders),
    "product_id": rng.integers(101, 106, n_orders),
    "quantity": rng.integers(1, 6, n_orders),
    "order_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 90, n_orders), unit="D")
})

# Sprinkle in a few missing values and a duplicate, like real data
orders.loc[3, "quantity"] = np.nan
orders = pd.concat([orders, orders.iloc[[5]]], ignore_index=True)   # duplicate row 5

print(orders.head())
print()
print(f"orders shape: {orders.shape}")

   order_id  customer_id  product_id  quantity order_date
0         1            1         101       5.0 2024-02-09
1         2            7         104       3.0 2024-03-01
2         3            6         104       1.0 2024-02-28
3         4            4         102       NaN 2024-02-12
4         5            4         101       4.0 2024-03-17

orders shape: (41, 5)


## Step 2 — Inspect and clean (notebook 04)

In [2]:
print(orders.info())
print()
print("Missing values per column:")
print(orders.isnull().sum())
print()
print("Duplicate rows:", orders.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     41 non-null     int64         
 1   customer_id  41 non-null     int64         
 2   product_id   41 non-null     int64         
 3   quantity     40 non-null     float64       
 4   order_date   41 non-null     datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(3)
memory usage: 1.7 KB
None

Missing values per column:
order_id       0
customer_id    0
product_id     0
quantity       1
order_date     0
dtype: int64

Duplicate rows: 1


In [3]:
orders_clean = orders.drop_duplicates().copy()
orders_clean["quantity"] = orders_clean["quantity"].fillna(orders_clean["quantity"].median())
orders_clean["quantity"] = orders_clean["quantity"].astype(int)

print("After cleaning:")
print("Missing values:", orders_clean.isnull().sum().sum())
print("Duplicate rows:", orders_clean.duplicated().sum())
print("Shape:", orders_clean.shape)

After cleaning:
Missing values: 0
Duplicate rows: 0
Shape: (40, 5)


## Step 3 — Merge the three tables into one analysis-ready DataFrame (notebook 07)

In [4]:
full = (
    orders_clean
    .merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
)

full["line_total"] = full["quantity"] * full["unit_price"]

print(full.head())
print()
print("Merged shape:", full.shape)

   order_id  customer_id  product_id  quantity order_date customer_name  \
0         1            1         101         5 2024-02-09         Alice   
1         2            7         104         3 2024-03-01        George   
2         3            6         104         1 2024-02-28         Fiona   
3         4            4         102         3 2024-02-12         Diana   
4         5            4         101         4 2024-03-17         Diana   

      city product_name category  unit_price  line_total  
0      NYC       Widget    Tools        9.99       49.95  
1  Houston    Doohickey    Tools       14.99       44.97  
2      NYC    Doohickey    Tools       14.99       14.99  
3  Chicago       Gadget     Tech       24.99       74.97  
4  Chicago       Widget    Tools        9.99       39.96  

Merged shape: (40, 11)


## Step 4 — Datetime features (notebook 08)

In [5]:
full["month"] = full["order_date"].dt.month_name()
full["weekday"] = full["order_date"].dt.day_name()
full["quarter"] = full["order_date"].dt.quarter

print(full[["order_date", "month", "weekday", "quarter"]].head())

  order_date     month    weekday  quarter
0 2024-02-09  February     Friday        1
1 2024-03-01     March     Friday        1
2 2024-02-28  February  Wednesday        1
3 2024-02-12  February     Monday        1
4 2024-03-17     March     Sunday        1


## Step 5 — Filtering and boolean indexing (notebooks 03 & 05)

In [6]:
big_orders = full[full["line_total"] > 50]
print(f"Orders over $50: {len(big_orders)} out of {len(full)}")
print(big_orders[["order_id", "customer_name", "product_name", "line_total"]].head())

nyc_tech_orders = full[(full["city"] == "NYC") & (full["category"] == "Tech")]
print(f"\nNYC customers buying Tech products: {len(nyc_tech_orders)} orders")

Orders over $50: 13 out of 40
    order_id customer_name product_name  line_total
3          4         Diana       Gadget       74.97
8          9           Bob    Doohickey       59.96
9         10         Alice    Doohickey       74.95
12        13         Fiona       Gadget      124.95
14        15         Fiona        Gizmo       99.95

NYC customers buying Tech products: 7 orders


## Step 6 — GroupBy analysis (notebook 06)

Which products and cities generate the most revenue?

In [7]:
revenue_by_product = full.groupby("product_name")["line_total"].agg(
    total_revenue="sum", num_orders="count", avg_order_value="mean"
).sort_values("total_revenue", ascending=False)

print("Revenue by product:")
print(revenue_by_product.round(2))

Revenue by product:
              total_revenue  num_orders  avg_order_value
product_name                                            
Gadget               574.77           7            82.11
Gizmo                519.74           9            57.75
Doohickey            419.72          11            38.16
Widget               299.70           8            37.46
Thingamajig           95.84           5            19.17


In [8]:
revenue_by_city = full.groupby("city")["line_total"].sum().sort_values(ascending=False)
print("Revenue by city:")
print(revenue_by_city.round(2))

Revenue by city:
city
NYC        686.62
LA         439.67
Houston    418.75
Chicago    364.73
Name: line_total, dtype: float64


In [9]:
# Monthly revenue trend
monthly_revenue = full.groupby("month")["line_total"].sum()
# reorder chronologically instead of alphabetically
month_order = ["January", "February", "March"]
monthly_revenue = monthly_revenue.reindex(month_order)
print("Monthly revenue:")
print(monthly_revenue.round(2))

Monthly revenue:
month
January     643.62
February    724.51
March       541.64
Name: line_total, dtype: float64


## Step 7 — Pivot table: category performance by city (notebook 09)

In [10]:
pivot = full.pivot_table(
    values="line_total",
    index="city",
    columns="category",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total"
)
print(pivot.round(2))

category   Misc     Tech   Tools    Total
city                                     
Chicago   29.95   154.93  179.85   364.73
Houston   23.96   269.88  124.91   418.75
LA        29.95   164.93  244.79   439.67
NYC       11.98   504.77  169.87   686.62
Total     95.84  1094.51  719.42  1909.77


## Step 8 — Customer-level summary (transform + groupby combined)

In [11]:
full["customer_total_spend"] = full.groupby("customer_id")["line_total"].transform("sum")

customer_summary = (
    full.groupby(["customer_id", "customer_name"])
    .agg(total_spent=("line_total", "sum"), num_orders=("order_id", "count"))
    .sort_values("total_spent", ascending=False)
    .reset_index()
)

print("Top customers by total spend:")
print(customer_summary.head())

Top customers by total spend:
   customer_id customer_name  total_spent  num_orders
0            7        George       418.75           9
1            6         Fiona       366.82           6
2            1         Alice       264.83           5
3            2           Bob       264.83           4
4            4         Diana       263.82           6


## Step 9 — Export the final analysis (notebook 09)

In [12]:
full.to_csv("retail_analysis_full.csv", index=False)
customer_summary.to_csv("customer_summary.csv", index=False)

print("Saved retail_analysis_full.csv and customer_summary.csv")

# Confirm the round-trip works
reloaded = pd.read_csv("retail_analysis_full.csv", parse_dates=["order_date"])
print(reloaded.dtypes[["order_date", "line_total"]])

Saved retail_analysis_full.csv and customer_summary.csv
order_date    datetime64[us]
line_total           float64
dtype: object


## Recap: where each concept showed up

| Concept | Where |
|---|---|
| **Creation & attributes** | Building `customers`, `products`, `orders` DataFrames |
| **Inspection & cleaning** | `.info()`, `.isnull().sum()`, `.duplicated()`, `.fillna()` |
| **Indexing & filtering** | `full[full["line_total"] > 50]`, combined boolean conditions |
| **GroupBy & aggregation** | Revenue by product/city/month, named aggregation, `.transform()` |
| **Merging** | Joining `orders` with `customers` and `products` |
| **Datetime handling** | `.dt.month_name()`, `.dt.day_name()`, `.dt.quarter` |
| **Pivot tables** | City × category revenue breakdown with margins |
| **I/O** | `.to_csv()` / `pd.read_csv()` with `parse_dates` |

### ✍️ Final challenge (extend this project yourself)

1. Add a `discount` column (e.g. random 0-20%) and recompute `line_total` after discount.
2. Find each customer's favorite category (the one they spend the most on).
3. Compute month-over-month revenue growth as a percentage.
4. Use `.resample()` (set `order_date` as the index first) to get weekly revenue totals and a
   4-week rolling average.

**You've now covered the core of pandas:**
Series/DataFrames → indexing & selection → cleaning → filtering/sorting/modifying →
groupby & aggregation → merging → datetime handling → I/O & pivoting — and combined them all
into a realistic, multi-table analysis. 🎉